# 29b · CXR-LT — đặc trưng EHR (Stage 1b)

Dựng bảng đặc trưng EHR mức study cho **227,835 study**, neo vào `index_time` của
`cxrlt_base.parquet`. Dùng `02_ehr_features.ipynb` làm **bản tham chiếu để port mẫu truy vấn**,
không chạy lại nguyên khối.

## Ba bất biến

```
1 DINH DANH : moi gia tri join qua subject_id, quy ve dung study_id cua no
2 THOI GIAN : labs / vitals / comorbidity phai thoa  storetime <= index_time
              (da RESULTED, khong chi duoc chi dinh) — khong mot giot tuong lai
3 Y NGHIA   : moi cua so thoi gian la MOT NHOM COT RIENG, khong gop trung binh.
              lab 24h = trang thai CAP; lab 90d = trang thai MAN — hai khai niem
              lam sang khac nhau, gop lai la mat nghia
```

## Ngoại lệ có căn cứ sinh học — microbiology

Cửa sổ vi sinh là `[index_time − 7d, index_time + 72h]` **trên giờ lấy bệnh phẩm** (`charttime`),
không phải giờ trả kết quả. Vi khuẩn **đã ở trong người bệnh ngay lúc chụp**; chỉ có *phép đo* bị
trễ 24–72h vì cần nuôi cấy. Chặn cứng ở `index_time` sẽ vứt bỏ nó vì lý do vận hành phòng xét
nghiệm, không phải vì lý do nhân quả.

Nhưng hai loại đại lượng khác hẳn nhau:

| đại lượng | bản chất | cửa sổ tương lai |
|---|---|---|
| **kết quả**: mọc ra con gì, dương/âm | trạng thái **sinh học**, đã tồn tại lúc chụp | **có** |
| **hành vi**: "có gửi bệnh phẩm không" | **hành động lâm sàng**, có thể do chính tấm phim gây ra | **không** — chặn ở `index_time` |

Cấy đờm gửi 6h *sau* một phim thấy đông đặc rất có thể được chỉ định **vì** phim đó. Lúc ấy đặc
trưng không mã hoá vi sinh mà mã hoá chẩn đoán của bác sĩ đọc phim — rò rỉ nhãn qua đường hành vi.

`MICRO_FUTURE_H` là công tắc, sweep `{0, 24, 48, 72}` ở Stage 2. Tăng đều theo cửa sổ = tín hiệu
nuôi cấy thật; **nhảy vọt ngay ở 24h rồi phẳng** = đang bắt hành vi chỉ định.

**CPU + BigQuery, ~1–2 giờ.** Cần `29_cxrlt_ehr_base.ipynb` đã chạy.

## C0 · Config

In [1]:
import os, json, time, re
import numpy as np, pandas as pd

PROJECT = 'mimic-ehr-v310'
HOSP    = 'physionet-data.mimiciv_3_1_hosp'
ICU     = 'physionet-data.mimiciv_3_1_icu'
ED      = None                      # giai o C1 — KHONG doan
TMP_DS  = 'cxrlt_tmp'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
P_BASE  = os.path.join(OUT_DIR, 'cxrlt_base.parquet')
assert os.path.exists(P_BASE), f'khong thay {P_BASE} -> chay 29_cxrlt_ehr_base.ipynb truoc'

# ---- cua so, theo bat bien 3: MOI cai la mot nhom cot rieng ----
LAB_WINDOWS = [('24h', 24), ('7d', 24*7), ('90d', 24*90)]   # cua so QUET
# Chi hai cua so dau cho ra COT GIA TRI. 90d chi giu SO DEM + DO MOI:
# lab 90 ngay khong con gia tri lam sang, va gia tri cua no thuc chat ma hoa
# 'co tiep xuc he thong y te' — tuc Q_CTX tra hinh, se lam ban Q_CLIN.
# Trang thai man da co comorbidity ICD (ckd/chf/cirrhosis) lam truc tiep hon.
LAB_VALUE_WINDOWS = [('24h', 24), ('7d', 24*7)]
MICRO_PAST_D   = 7      # ngay truoc khi chup (lay benh pham)
MICRO_FUTURE_H = 72     # gio sau khi chup — sweep {0,24,48,72} o Stage 2
VITAL_WINDOW_H = 24
ORG_TOP_K      = 30     # so organism duoc one-hot; vocab khop CHI TREN TRAIN

BASE = pd.read_parquet(P_BASE)
print(f'{len(BASE):,} study | setting {BASE.setting.value_counts().to_dict()}')
print(f'index_time {BASE.index_time.min()} -> {BASE.index_time.max()}')
assert BASE.study_id.is_unique and BASE.index_time.notna().all()

Mounted at /content/drive
227,835 study | setting {'inpatient': 110892, 'ed': 77801, 'outpatient': 39142}
index_time 2110-01-11 22:01:11.359000+00:00 -> 2208-12-08 12:04:54.936000+00:00


## C0-bis · Đăng nhập BigQuery bằng `gcloud`

`auth.authenticate_user()` không dùng được — tài khoản không phải tài khoản Cloud. Chạy cell dưới,
mở link, **dán mã trả về ngược lại ô nhập**. Phải làm lại **mỗi phiên Colab**.

In [2]:
!gcloud auth application-default login --no-launch-browser

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=w1CvZTRqNqjLY4BHRX0CTBxIvxZFHF&prompt=consent&token_usage=remote&access_type=offline&code_challenge=aJoQuCJTLRb-sjRgip04LYnKZ0FbYPZFVvwGEfqgStU&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AXEQxICFYFdrIQ45OkFrZkQLm_kmyWRPnAPuafZerP0i0wNhB5OLVX6LdmqGse7sydMtlQ

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

### C0-ter · Gán project + `run_q`

In [3]:
!gcloud config set project {PROJECT}
!gcloud auth application-default set-quota-project {PROJECT} 2>/dev/null || true

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location='US')

def run_q(sql, label='', poll=2.0):
    '''Dry-run truoc (0 byte tinh tien, bat loi go phim), roi moi chay that.

    Hien tien trinh cho CA HAI pha cua mot truy van BigQuery:
      1. CHAY tren cluster -> khong stream duoc, nhung poll duoc trang thai va
         so byte da quet; in tai cho moi `poll` giay.
      2. TAI ket qua ve    -> client ho tro progress_bar_type='tqdm'.
    '''
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False))
    est = dry.total_bytes_processed / 1e9
    print(f'  [{label}] uoc quet {est:.2f} GB, bat dau...')

    t0 = time.time()
    job = client.query(sql)                       # KHONG chan — tra ve ngay
    while not job.done():                         # done() tu poll server
        done_gb = (job.total_bytes_processed or 0) / 1e9
        pct = f'{done_gb/est:5.0%}' if est > 0 else '  n/a'
        print(f'  [{label}] {job.state:<7} {pct} ({done_gb:5.2f}/{est:.2f} GB) '
              f'· {time.time()-t0:5.0f}s', end='\r')
        time.sleep(poll)
    t_run = time.time() - t0

    try:
        df = job.to_dataframe(progress_bar_type='tqdm')      # thanh tien trinh khi TAI
    except Exception:
        df = job.to_dataframe()                              # tqdm thieu -> im lang
    act = (job.total_bytes_processed or 0) / 1e9
    print(f'  [{label}] {act:5.2f} GB thuc · chay {t_run:5.0f}s · tai '
          f'{time.time()-t0-t_run:4.0f}s · {len(df):>9,} dong')
    return df

print('BigQuery OK ->', [dict(r) for r in client.query('SELECT 1 AS ok').result()])


To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Updated property [core/project].
BigQuery OK -> [{'ok': 1}]


## C1 · Giải tên ED + đẩy bảng khoá `coh` lên BigQuery

In [4]:
# ED phai khop phien ban voi HOSP neu co; khong thi lay ban duy nhat va NOI RO
all_ds = [d.dataset_id for d in client.list_datasets('physionet-data')]
cands  = [d for d in all_ds if d.startswith('mimiciv') and d.split('_')[-1] == 'ed']
_want  = HOSP.split('.')[-1][:-len('_hosp')] + '_ed'
ED = f'physionet-data.{_want}' if _want in cands else f'physionet-data.{sorted(cands)[-1]}'
print(f'ED -> {ED}' + ('' if _want in cands else f'   (!! khong co {_want}; ung vien {cands})'))

ds = f'{PROJECT}.{TMP_DS}'
client.create_dataset(bigquery.Dataset(ds), exists_ok=True)
COH = f'`{ds}.coh`'

keys = BASE[['study_id', 'subject_id', 'index_time', 'cur_hadm', 'cur_edstay',
             'setting', 'split']].copy()
keys['index_time'] = pd.to_datetime(keys.index_time)
client.load_table_from_dataframe(
    keys, f'{ds}.coh',
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()
_n = client.query(f'SELECT COUNT(*) n FROM {COH}').to_dataframe().n[0]
assert int(_n) == len(keys), f'BigQuery co {_n} dong, local co {len(keys)}'
print(f'da day -> {ds}.coh  ({int(_n):,} study)')

TRAIN_IDS = set(BASE.loc[BASE.split == 'train', 'study_id'])   # vocab CHI khop tren day
print(f'study train: {len(TRAIN_IDS):,}')

ED -> physionet-data.mimiciv_ed   (!! khong co mimiciv_3_1_ed; ung vien ['mimiciv_ed'])
da day -> mimic-ehr-v310.cxrlt_tmp.coh  (227,835 study)
study train: 155,823


## C1-bis · Module ED có phủ hết dải thời gian không?

`mimiciv_3_1_ed` **không tồn tại** — chỉ có `mimiciv_ed`, được đánh phiên bản độc lập với
MIMIC-IV. Nếu nó phủ khoảng thời gian **hẹp hơn** `hosp`, thì study nằm ngoài khoảng đó sẽ không
khớp `edstay` nào và **bị gán nhầm thành `outpatient`** — thổi phồng nhóm 17.18% một cách giả tạo,
và làm mọi phân tích tách theo `setting` sai lệch.

**Không được gom theo năm của `index_time`.** MIMIC-IV dịch chuyển ngày tháng **theo từng bệnh
nhân**, nên năm đó là năm đã dịch; gom theo nó sẽ trộn lẫn các thời kỳ thật và che mất đúng ranh
giới ta đang tìm.

Cách đúng là dựng lại **năm thật** qua `anchor_year_group` — cùng thủ thuật `26_add_demographics`
dùng để tính tuổi:

```
nam_that ~ dau(anchor_year_group) + (year(index_time) - anchor_year)
```

Nếu tỉ lệ `ed` tụt về gần 0 ở một đầu dải năm thật, phần `outpatient` ở đó là **giả**.

In [5]:
edc = run_q(f'''
WITH yr AS (
  SELECT c.study_id, c.setting,
         CAST(SUBSTR(p.anchor_year_group, 1, 4) AS INT64)
           + (EXTRACT(YEAR FROM c.index_time) - p.anchor_year) AS nam_that
  FROM {COH} c JOIN `{HOSP}.patients` p ON p.subject_id = c.subject_id
)
SELECT nam_that, setting, COUNT(*) n FROM yr GROUP BY nam_that, setting
''', 'ed-coverage')

piv = (edc.pivot(index='nam_that', columns='setting', values='n')
          .reindex(columns=['inpatient', 'ed', 'outpatient']).fillna(0).astype(int))
piv = piv[piv.sum(1) >= 200]                    # bo nam qua it de doc
frac = piv.div(piv.sum(1), axis=0)
out = piv.assign(tong=piv.sum(1))
for c in ['inpatient', 'ed', 'outpatient']:
    out[f'%{c[:3]}'] = frac[c].map('{:.1%}'.format)
print('=== SETTING theo NAM THAT (dung anchor_year_group, khong dung nam da dich) ===')
print(out[['tong', '%inp', '%ed', '%out']].to_string())

_ed = frac['ed']
_med = _ed.median()
_bad = _ed[_ed < _med * 0.5]                    # nam co ty le ed thap bat thuong
print(f'\nty le `ed` trung vi qua cac nam: {_med:.1%}')
if len(_bad):
    _n = int(piv.loc[_bad.index].sum(1).sum())
    _o = int(piv.loc[_bad.index, 'outpatient'].sum())
    print(f'!! {len(_bad)} nam co ty le `ed` < mot nua trung vi: {list(_bad.index)}')
    print(f'   {_n:,} study nam trong cac nam do, trong do {_o:,} bi gan `outpatient`')
    print(f'   -> mot phan trong so {_o:,} nay CO THE la ED that nhung ngoai vung phu cua module.')
    print(f'      Khi bao cao tach theo setting, ghi ro con so nay.')
else:
    print('OK  ty le `ed` on dinh qua cac nam -> module ED phu du dai,')
    print('    nhom `outpatient` khong bi thoi phong vi thieu du lieu ED.')

# --- chung thuc bang chinh dai thoi gian cua hai bang ---
rng = run_q(f'''
SELECT 'edstays' AS bang, MIN(CAST(intime AS TIMESTAMP)) AS tu,
       MAX(CAST(intime AS TIMESTAMP)) AS den, COUNT(DISTINCT subject_id) AS n_subject
FROM `{ED}.edstays`
UNION ALL
SELECT 'admissions', MIN(CAST(admittime AS TIMESTAMP)),
       MAX(CAST(admittime AS TIMESTAMP)), COUNT(DISTINCT subject_id)
FROM `{HOSP}.admissions`
''', 'module-range')
print('\n=== DAI THOI GIAN (da dich) VA SO BENH NHAN CUA HAI MODULE ===')
print(rng.to_string(index=False))
print('   (dai thoi gian da dich khong so truc tiep duoc; cot n_subject moi la thong tin:')
print('    neu edstays it benh nhan hon admissions nhieu, do la gioi han phu song that.)')

  [ed-coverage] uoc quet 0.02 GB, bat dau...
Job ID e52f0eee-0568-4078-80d3-cd6ac5e00d49 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [ed-coverage]  0.02 GB thuc · chay     1s · tai    1s ·        28 dong
=== SETTING theo NAM THAT (dung anchor_year_group, khong dung nam da dich) ===
setting    tong   %inp    %ed   %out
nam_that                            
2009       1042  36.9%  39.3%  23.7%
2010       4716  35.9%  38.4%  25.7%
2011      48266  55.6%  35.0%   9.4%
2012      28055  45.7%  37.5%  16.8%
2013      23675  41.2%  40.0%  18.9%
2014      72552  57.5%  33.7%   8.8%
2015      24541  47.8%  37.1%  15.2%
2016      11217  45.3%  39.1%  15.6%
2017       1932  46.9%  39.2%  13.9%

ty le `ed` trung vi qua cac nam: 38.4%
OK  ty le `ed` on dinh qua cac nam -> module ED phu du dai,
    nhom `outpatient` khong bi thoi phong vi thieu du lieu ED.
  [module-range] uoc quet 0.02 GB, bat dau...
Job ID 26a287e2-82ed-49e3-97cf-1ca36019f5bd successfully executed: 100%|█

## C2 · Bảng lab — **kiểm khớp trước khi tin**

Mở rộng panel của nb02 (7 xét nghiệm) lên 18, thêm nhóm liên quan trực tiếp tới bệnh tim–phổi mà
40 nhãn của CXR-LT có nhiều: **NT-proBNP** ↔ suy tim (`Cardiomegaly`, `Edema`), **troponin** ↔
thiếu máu cơ tim, **D-dimer** ↔ thuyên tắc phổi (`Pulmonary Embolism`), **lactate** ↔ giảm tưới máu.

`fluid='Blood'` là điều kiện **chịu lực**, không phải trang trí: `'Red Blood Cells'` cũng tồn tại
trong **nước tiểu** (tổng phân tích nước tiểu), đếm hồng cầu niệu thành công thức máu là rác âm thầm.

Cell này in ra **`d_labitems` thật sự khớp cái gì** rồi mới đi tiếp — nếu một canon không khớp
itemid nào thì mẫu chuỗi sai, không phải dữ liệu thiếu.

In [6]:
LAB_CASE = '''
    WHEN LOWER(label) LIKE '%urea nitrogen%'      AND fluid='Blood' THEN 'lab_bun'
    WHEN LOWER(label) = 'creatinine'              AND fluid='Blood' THEN 'lab_creatinine'
    WHEN LOWER(label) = 'glucose'                 AND fluid='Blood' THEN 'lab_glucose'
    WHEN LOWER(label) = 'hemoglobin'              AND fluid='Blood' THEN 'lab_hemoglobin'
    WHEN LOWER(label) = 'red blood cells'         AND fluid='Blood' THEN 'lab_rbc'
    WHEN LOWER(label) LIKE '%platelet count%'                       THEN 'lab_platelet'
    WHEN LOWER(label) LIKE '%white blood cell%'                     THEN 'lab_wbc'
    WHEN LOWER(label) = 'sodium'                  AND fluid='Blood' THEN 'lab_sodium'
    WHEN LOWER(label) = 'potassium'               AND fluid='Blood' THEN 'lab_potassium'
    WHEN LOWER(label) = 'chloride'                AND fluid='Blood' THEN 'lab_chloride'
    WHEN LOWER(label) = 'bicarbonate'             AND fluid='Blood' THEN 'lab_bicarbonate'
    WHEN LOWER(label) = 'albumin'                 AND fluid='Blood' THEN 'lab_albumin'
    WHEN LOWER(label) LIKE '%bilirubin, total%'   AND fluid='Blood' THEN 'lab_bilirubin'
    WHEN LOWER(label) LIKE '%inr%'                                  THEN 'lab_inr'
    WHEN LOWER(label) = 'lactate'                 AND fluid='Blood' THEN 'lab_lactate'
    WHEN LOWER(label) LIKE '%troponin t%'                           THEN 'lab_troponin'
    WHEN LOWER(label) LIKE '%ntprobnp%'                             THEN 'lab_ntprobnp'
    WHEN LOWER(label) LIKE '%d-dimer%'                              THEN 'lab_ddimer'
'''
LABS = sorted(set(re.findall(r"THEN '(lab_\w+)'", LAB_CASE)))
print(f'{len(LABS)} xet nghiem: {LABS}')

chk = run_q(f'''
SELECT canon, COUNT(*) n_itemid, STRING_AGG(DISTINCT label ORDER BY label LIMIT 8) vi_du
FROM (SELECT itemid, label, CASE {LAB_CASE} END AS canon FROM `{HOSP}.d_labitems`)
WHERE canon IS NOT NULL GROUP BY canon ORDER BY canon
''', 'lab-map')
print(chk.to_string(index=False))
_miss = sorted(set(LABS) - set(chk.canon))
assert not _miss, (f'KHONG khop itemid nao: {_miss}\n'
                   f'  -> mau chuoi sai, khong phai du lieu thieu. Sua LAB_CASE.')

# --- CONG: mot canon khop QUA NHIEU itemid nghia la mau chuoi qua rong ---
# Ta lay gia tri GAN NHAT theo canon. Neu canon gom nhieu dai luong khac chat
# (vd 'Hemoglobin' g/dL lan 'Hemoglobin A1c' %), mot so study se nhan gia tri cua
# dai luong SAI ma khong co gi bao. Do la rac am tham, khong phai thieu du lieu.
MAX_ITEMID = 4
_fat = chk[chk.n_itemid > MAX_ITEMID]
if len(_fat):
    print(f'\n!! {len(_fat)} canon khop > {MAX_ITEMID} itemid — mau chuoi qua rong:')
    print(_fat.to_string(index=False))
assert not len(_fat), (
    f'canon qua rong: {list(_fat.canon)}. Siet mau trong LAB_CASE '
    f'(dung `= \'ten chinh xac\'` thay cho `LIKE \'%tu khoa%\'`) roi chay lai C2.')

_multi = chk[chk.n_itemid >= 3]
if len(_multi):
    print(f'\n{len(_multi)} canon khop 3-{MAX_ITEMID} itemid — NHIN QUA xem cung dai luong khong:')
    print(_multi.to_string(index=False))
print(f'\nca {len(LABS)} canon deu khop >=1 itemid va khong canon nao > {MAX_ITEMID}')

18 xet nghiem: ['lab_albumin', 'lab_bicarbonate', 'lab_bilirubin', 'lab_bun', 'lab_chloride', 'lab_creatinine', 'lab_ddimer', 'lab_glucose', 'lab_hemoglobin', 'lab_inr', 'lab_lactate', 'lab_ntprobnp', 'lab_platelet', 'lab_potassium', 'lab_rbc', 'lab_sodium', 'lab_troponin', 'lab_wbc']
  [lab-map] uoc quet 0.00 GB, bat dau...
Job ID 4b805196-ef34-4fcb-985d-fc16de905f83 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [lab-map]  0.00 GB thuc · chay     1s · tai    1s ·        18 dong
          canon  n_itemid             vi_du
    lab_albumin         2           Albumin
lab_bicarbonate         1       Bicarbonate
  lab_bilirubin         2  Bilirubin, Total
        lab_bun         2     Urea Nitrogen
   lab_chloride         2          Chloride
 lab_creatinine         2        Creatinine
     lab_ddimer         3           D-Dimer
    lab_glucose         3           Glucose
 lab_hemoglobin         3        Hemoglobin
        lab_inr         2           INR(PT)
    la

## C3 · Labs — ba cửa sổ thành **ba nhóm cột riêng**

Với mỗi `(study, xét nghiệm)`, lấy **giá trị mới nhất** trong từng cửa sổ, kèm `delta_h` (giờ
trước lúc chụp) và **số lần đo**.

**Cửa sổ 90 ngày chỉ cho ra số đếm và độ mới, KHÔNG cho ra giá trị.** Lab cấp phân rã theo
giờ, nên một giá trị 90 ngày trước không mô tả trạng thái hôm nay; phần *trạng thái mạn* đã
có comorbidity ICD (`ckd`, `chf`, `cirrhosis`) làm trực tiếp hơn. Quan trọng hơn cả: một giá
trị lab 90d thực chất mã hoá *"bệnh nhân có tiếp xúc hệ thống y tế"* — tức **sử dụng dịch vụ**,
nhưng lại bị quy tắc phân tier xếp vào `Q_CLIN`. Đó là cấy một đặc trưng `Q_CTX` trá hình vào
`Q_CLIN`, làm hỏng chính phép đo `Q_CLIN − Q_CTX`. Số đếm thì giữ, vì nó là đặc trưng sử dụng
dịch vụ **trung thực** và nằm đúng chỗ ở `Q_CTX`. Số lần đo là đặc trưng `Q_CTX` — nó mã hoá cường độ theo dõi,
không mã hoá giá trị lâm sàng.

Bất biến 2 nằm ở đúng một dòng: `lr.st <= c.index_time` — **đã có kết quả**, không phải mới chỉ định.

In [7]:
_win_sel = ',\n  '.join(
    f'''ARRAY_AGG(IF(delta_h <= {h}, valuenum, NULL) IGNORE NULLS
              ORDER BY ct DESC LIMIT 1)[SAFE_OFFSET(0)] AS v_{tag},
  MIN(IF(delta_h <= {h}, delta_h, NULL))                AS dh_{tag},
  COUNTIF(delta_h <= {h})                               AS n_{tag}'''
    for tag, h in LAB_WINDOWS)
_maxh = max(h for _, h in LAB_WINDOWS)

labq = run_q(f'''
WITH lab_map AS (
  SELECT itemid, CASE {LAB_CASE} END AS canon FROM `{HOSP}.d_labitems`
),
lr AS (
  SELECT le.subject_id, lm.canon, le.valuenum,
         CAST(le.charttime AS TIMESTAMP) AS ct,
         COALESCE(CAST(le.storetime AS TIMESTAMP),
                  CAST(le.charttime AS TIMESTAMP)) AS st
  FROM `{HOSP}.labevents` le JOIN lab_map lm USING (itemid)
  WHERE lm.canon IS NOT NULL AND le.valuenum IS NOT NULL AND le.charttime IS NOT NULL
),
j AS (
  SELECT c.study_id, lr.canon, lr.valuenum, lr.ct,
         TIMESTAMP_DIFF(c.index_time, lr.ct, MINUTE)/60.0 AS delta_h
  FROM {COH} c JOIN lr ON lr.subject_id = c.subject_id
   AND lr.ct BETWEEN TIMESTAMP_SUB(c.index_time, INTERVAL {_maxh} HOUR) AND c.index_time
   AND lr.st <= c.index_time              -- BAT BIEN 2: da co ket qua truoc luc chup
)
SELECT study_id, canon, {_win_sel}
FROM j GROUP BY study_id, canon
''', 'labs')

assert (labq.dh_24h.dropna() >= 0).all(), 'LEAK: lab do SAU index_time'
print(f'\ndo phu (co it nhat 1 xet nghiem trong cua so):')
for tag, _ in LAB_WINDOWS:
    _any = labq.groupby('study_id')[f'n_{tag}'].sum().gt(0).reindex(BASE.study_id).fillna(False)
    print(f'  {tag:>4}: {_any.mean():7.2%}')

  [labs] uoc quet 6.15 GB, bat dau...
Job ID 455cf2c1-be3c-467b-8cf6-95c8354ab4eb successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [labs]  6.15 GB thuc · chay    10s · tai   13s · 2,474,488 dong

do phu (co it nhat 1 xet nghiem trong cua so):
   24h:  64.41%
    7d:  70.34%
   90d:  81.11%


## C4 · Vitals — **họ duy nhất bị encounter chặn**

In [8]:
vitq = run_q(f'''
WITH vi AS (
  SELECT c.subject_id, CAST(c.charttime AS TIMESTAMP) AS ct,
         CASE c.itemid WHEN 223761 THEN (c.valuenum-32)*5/9 WHEN 223762 THEN c.valuenum END AS temp_c,
         IF(c.itemid = 220045, c.valuenum, NULL) AS hr,
         IF(c.itemid = 220210, c.valuenum, NULL) AS rr,
         IF(c.itemid = 220277, c.valuenum, NULL) AS spo2,
         IF(c.itemid IN (220179, 220050), c.valuenum, NULL) AS sbp,
         IF(c.itemid IN (220180, 220051), c.valuenum, NULL) AS dbp
  FROM `{ICU}.chartevents` c
  WHERE c.itemid IN (223761,223762,220045,220210,220277,220179,220050,220180,220051)
    AND c.valuenum IS NOT NULL
  UNION ALL
  SELECT subject_id, CAST(charttime AS TIMESTAMP), (temperature-32)*5/9,
         heartrate, resprate, o2sat, sbp, dbp
  FROM `{ED}.vitalsign`
),
vv AS (
  SELECT * FROM vi WHERE COALESCE(temp_c,hr,rr,spo2,sbp,dbp) IS NOT NULL
),
j AS (
  SELECT c.study_id, vv.*, TIMESTAMP_DIFF(c.index_time, vv.ct, MINUTE)/60.0 AS delta_h
  FROM {COH} c JOIN vv ON vv.subject_id = c.subject_id
   AND vv.ct BETWEEN TIMESTAMP_SUB(c.index_time, INTERVAL {VITAL_WINDOW_H} HOUR)
                 AND c.index_time          -- BAT BIEN 2
)
SELECT study_id,
  ARRAY_AGG(temp_c IGNORE NULLS ORDER BY ct DESC LIMIT 1)[SAFE_OFFSET(0)] AS vital_temp_c,
  ARRAY_AGG(hr     IGNORE NULLS ORDER BY ct DESC LIMIT 1)[SAFE_OFFSET(0)] AS vital_hr,
  ARRAY_AGG(rr     IGNORE NULLS ORDER BY ct DESC LIMIT 1)[SAFE_OFFSET(0)] AS vital_rr,
  ARRAY_AGG(spo2   IGNORE NULLS ORDER BY ct DESC LIMIT 1)[SAFE_OFFSET(0)] AS vital_spo2,
  ARRAY_AGG(sbp    IGNORE NULLS ORDER BY ct DESC LIMIT 1)[SAFE_OFFSET(0)] AS vital_sbp,
  ARRAY_AGG(dbp    IGNORE NULLS ORDER BY ct DESC LIMIT 1)[SAFE_OFFSET(0)] AS vital_dbp,
  COUNT(*) AS vital_n_meas, MIN(delta_h) AS vital_delta_h
FROM j GROUP BY study_id
''', 'vitals')

# khoang sinh ly — gia tri ngoai khoang la loi don vi, phai thanh NaN chu khong duoc giu
VIT_RANGE = dict(vital_temp_c=(25,45), vital_hr=(10,300), vital_rr=(2,80),
                 vital_spo2=(20,100), vital_sbp=(30,300), vital_dbp=(10,200))
for c, (lo, hi) in VIT_RANGE.items():
    bad = ((vitq[c] < lo) | (vitq[c] > hi)).sum()
    vitq.loc[(vitq[c] < lo) | (vitq[c] > hi), c] = np.nan
    if bad: print(f'  {c}: {bad:,} gia tri ngoai [{lo},{hi}] -> NaN')

_cov = BASE[['study_id','setting']].merge(vitq[['study_id']].assign(co=1), how='left')
print('\ndo phu vitals theo setting (day la cho outpatient mat):')
print(_cov.groupby('setting').co.agg(n='size', co_vital=lambda s: s.notna().sum())
        .assign(ty_le=lambda d: (d.co_vital/d.n).map('{:.2%}'.format)).to_string())

  [vitals] uoc quet 11.89 GB, bat dau...
Job ID 9d49e283-5e4c-49b8-b9e8-16f723b8db64 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [vitals] 11.89 GB thuc · chay     7s · tai    2s ·   114,608 dong
  vital_temp_c: 279 gia tri ngoai [25,45] -> NaN
  vital_hr: 6 gia tri ngoai [10,300] -> NaN
  vital_rr: 189 gia tri ngoai [2,80] -> NaN
  vital_spo2: 31 gia tri ngoai [20,100] -> NaN
  vital_sbp: 60 gia tri ngoai [30,300] -> NaN
  vital_dbp: 39 gia tri ngoai [10,200] -> NaN

do phu vitals theo setting (day la cho outpatient mat):
                 n  co_vital   ty_le
setting                             
ed           77801     46616  59.92%
inpatient   110892     67797  61.14%
outpatient   39142       195   0.50%


## C5 · Comorbidity — **neo vào lần nhập viện ĐÃ ĐÓNG trước đó**

Đây là họ đặc trưng **ngoại trú vẫn có**, nên đáng đầu tư nhất. Nó không bị encounter hiện tại chặn:
chỉ cần bệnh nhân từng nhập viện trước `index_time` và đợt đó **đã xuất viện** (`dischtime <
index_time`) — nếu lấy cả đợt đang mở thì mã ICD của chính đợt này rò rỉ vào.

Mở rộng từ 2 khái niệm của nb02 lên 10. Cell kiểm mẫu ICD khớp mã thật trước khi dùng — mẫu sai
thì im lặng cho ra toàn 0, nhìn y như "bệnh nhân không có bệnh nền".

In [9]:
COMO = {
 'diabetes': ("icd_code LIKE '250%'",
              "icd_code LIKE 'E08%' OR icd_code LIKE 'E09%' OR icd_code LIKE 'E10%'"
              " OR icd_code LIKE 'E11%' OR icd_code LIKE 'E13%'"),
 'hiv':      ("icd_code LIKE '042%' OR icd_code = 'V08' OR icd_code LIKE '07953%'",
              "icd_code LIKE 'B20%' OR icd_code = 'Z21'"),
 'copd':     ("icd_code LIKE '491%' OR icd_code LIKE '492%' OR icd_code LIKE '496%'",
              "icd_code LIKE 'J41%' OR icd_code LIKE 'J42%' OR icd_code LIKE 'J43%'"
              " OR icd_code LIKE 'J44%'"),
 'chf':      ("icd_code LIKE '428%'", "icd_code LIKE 'I50%'"),
 'ckd':      ("icd_code LIKE '585%'", "icd_code LIKE 'N18%'"),
 'cancer':   ("icd_code BETWEEN '140' AND '2099'",
              "icd_code LIKE 'C%' OR icd_code LIKE 'D0%'"),
 'cirrhosis':("icd_code LIKE '571%'", "icd_code LIKE 'K70%' OR icd_code LIKE 'K74%'"),
 'smoking':  ("icd_code LIKE '3051%' OR icd_code = 'V1582'",
              "icd_code LIKE 'F17%' OR icd_code = 'Z87891'"),
 'tb':       ("icd_code LIKE '010%' OR icd_code LIKE '011%' OR icd_code LIKE '012%'",
              "icd_code LIKE 'A15%' OR icd_code LIKE 'A16%' OR icd_code LIKE 'A17%'"
              " OR icd_code LIKE 'A18%' OR icd_code LIKE 'A19%'"),
 'pneumonia_hx': ("icd_code LIKE '480%' OR icd_code LIKE '481%' OR icd_code LIKE '482%'"
                  " OR icd_code LIKE '485%' OR icd_code LIKE '486%'",
                  "icd_code LIKE 'J12%' OR icd_code LIKE 'J13%' OR icd_code LIKE 'J14%'"
                  " OR icd_code LIKE 'J15%' OR icd_code LIKE 'J18%'"),
}
_pred = lambda k: (f"((icd_version = 9 AND ({COMO[k][0]})) OR "
                   f" (icd_version = 10 AND ({COMO[k][1]})))")

seen = run_q(f'''
SELECT concept, icd_version, COUNT(DISTINCT icd_code) n_ma, COUNT(*) n_dong
FROM ({" UNION ALL ".join(f"SELECT '{k}' AS concept, icd_version, icd_code "
                          f"FROM `{HOSP}.diagnoses_icd` WHERE {_pred(k)}" for k in COMO)})
GROUP BY concept, icd_version ORDER BY concept, icd_version
''', 'icd-verify')
print(seen.to_string(index=False))
_bad = [k for k in COMO if not ((seen.concept == k)).any()]
assert not _bad, f'KHONG khop ma ICD nao: {_bad} -> mau sai, khong phai du lieu thieu'
print(f'\nca {len(COMO)} khai niem deu khop ma that')

_sel = ',\n  '.join(f"MAX(IF({_pred(k)}, 1, 0)) AS como_{k}" for k in COMO)
como = run_q(f'''
WITH prior_adm AS (       -- CHI dot da XUAT VIEN truoc luc chup
  SELECT c.study_id, a.hadm_id
  FROM {COH} c JOIN `{HOSP}.admissions` a ON a.subject_id = c.subject_id
  WHERE CAST(a.dischtime AS TIMESTAMP) < c.index_time
)
SELECT p.study_id, COUNT(DISTINCT p.hadm_id) AS como_n_prior_adm,
  {_sel}
FROM prior_adm p JOIN `{HOSP}.diagnoses_icd` d USING (hadm_id)
GROUP BY p.study_id
''', 'comorbidity')
print(f'\nstudy co tien su nhap vien: {len(como):,} / {len(BASE):,} '
      f'({len(como)/len(BASE):.1%})')
print(como[[c for c in como.columns if c.startswith('como_') and c != 'como_n_prior_adm']]
        .mean().sort_values(ascending=False).map('{:.2%}'.format).to_string())

  [icd-verify] uoc quet 0.09 GB, bat dau...
Job ID a5009b05-aaad-45b5-b8ff-ba2df9fb85d9 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [icd-verify]  0.09 GB thuc · chay     4s · tai    2s ·        20 dong
     concept  icd_version  n_ma  n_dong
      cancer            9   533   59334
      cancer           10   833   68533
         chf            9    15   69023
         chf           10    23   43913
   cirrhosis            9    11   15538
   cirrhosis           10    20   16016
         ckd            9     7   39560
         ckd           10    10   43672
        copd            9    10   20076
        copd           10    12   25186
    diabetes            9    40   78987
    diabetes           10   196  101333
         hiv            9     3    5404
         hiv           10     2    4310
pneumonia_hx            9    25   14461
pneumonia_hx           10    28   15518
     smoking            9     2   57584
     smoking           10    17   91527
          

## C6 · Demographics — `age` + `gender`

In [10]:
demo = run_q(f'''
SELECT c.study_id,
       p.anchor_age + (EXTRACT(YEAR FROM c.index_time) - p.anchor_year) AS age,
       p.gender
FROM {COH} c JOIN `{HOSP}.patients` p ON p.subject_id = c.subject_id
''', 'demographics')
demo['age'] = demo.age.clip(0, 100)
print(f'co age/gender: {len(demo):,} / {len(BASE):,} ({len(demo)/len(BASE):.2%})')
print(f'  age: {demo.age.min():.0f} - {demo.age.max():.0f}, trung vi {demo.age.median():.0f}')
print(f'  gender: {demo.gender.value_counts().to_dict()}')
print(f'\n!! {len(BASE)-len(demo):,} study KHONG co (subject khong nam trong hosp.patients)')
print('   -> se de NaN + cot mask, KHONG dien gia tri trung binh')

  [demographics] uoc quet 0.02 GB, bat dau...
Job ID ceeb8d68-1be2-4e1b-a9dd-31d8a9d64f75 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [demographics]  0.02 GB thuc · chay     1s · tai    2s ·   215,998 dong
co age/gender: 215,998 / 227,835 (94.80%)
  age: 16 - 100, trung vi 65
  gender: {'M': 110890, 'F': 105108}

!! 11,837 study KHONG co (subject khong nam trong hosp.patients)
   -> se de NaN + cot mask, KHONG dien gia tri trung binh


## C7 · Microbiology — kết quả được cửa sổ tương lai, hành vi thì không

Hai truy vấn tách hẳn nhau, đúng phân biệt đã nêu ở đầu notebook:

* `micro_sent_*` — **hành vi**: đã gửi bệnh phẩm, `charttime <= index_time`. Không tương lai.
* `micro_pos_*`, `micro_org_*` — **kết quả**: cửa sổ `[−7d, +MICRO_FUTURE_H]` trên `charttime`.

Đặt `MICRO_FUTURE_H = 0` để có bản nhân quả chặt làm control.

In [11]:
sent = run_q(f'''
SELECT c.study_id,
       COUNT(*) AS micro_sent_n,
       COUNT(DISTINCT m.spec_itemid) AS micro_sent_n_spec,
       MIN(TIMESTAMP_DIFF(c.index_time, CAST(m.charttime AS TIMESTAMP), HOUR)) AS micro_sent_delta_h
FROM {COH} c JOIN `{HOSP}.microbiologyevents` m ON m.subject_id = c.subject_id
WHERE CAST(m.charttime AS TIMESTAMP)
        BETWEEN TIMESTAMP_SUB(c.index_time, INTERVAL {MICRO_PAST_D} DAY)
            AND c.index_time                      -- HANH VI: khong tuong lai
GROUP BY c.study_id
''', 'micro-sent')
assert (sent.micro_sent_delta_h >= 0).all(), 'LEAK: benh pham gui SAU index_time'

_hi = ('c.index_time' if MICRO_FUTURE_H == 0 else
       f'TIMESTAMP_ADD(c.index_time, INTERVAL {MICRO_FUTURE_H} HOUR)')
res = run_q(f'''
SELECT c.study_id, m.org_itemid,
       COUNTIF(m.org_itemid IS NOT NULL) OVER (PARTITION BY c.study_id) AS micro_pos_n
FROM {COH} c JOIN `{HOSP}.microbiologyevents` m ON m.subject_id = c.subject_id
WHERE CAST(m.charttime AS TIMESTAMP)
        BETWEEN TIMESTAMP_SUB(c.index_time, INTERVAL {MICRO_PAST_D} DAY) AND {_hi}
''', f'micro-result(+{MICRO_FUTURE_H}h)')

# vocab organism khop CHI TREN TRAIN — giong nb02 C17
_tr = res[res.study_id.isin(TRAIN_IDS) & res.org_itemid.notna()]
ORG_VOCAB = _tr.org_itemid.value_counts().head(ORG_TOP_K).index.tolist()
print(f'\norganism vocab: top {len(ORG_VOCAB)} khop tren {len(_tr):,} dong TRAIN')
print(f'   phu {_tr.org_itemid.isin(ORG_VOCAB).mean():.1%} so lan mo tren train')

micro = res.groupby('study_id').agg(
    micro_pos_n=('org_itemid', lambda s: int(s.notna().sum())),
    micro_n_org=('org_itemid', 'nunique')).reset_index()
for oid in ORG_VOCAB:
    micro[f'micro_org_{int(oid)}'] = micro.study_id.map(
        res[res.org_itemid == oid].groupby('study_id').size()).fillna(0).astype('int8')
micro = micro.merge(sent, on='study_id', how='outer')
print(f'study co ban ghi vi sinh: {len(micro):,} / {len(BASE):,} ({len(micro)/len(BASE):.1%})')

  [micro-sent] uoc quet 0.10 GB, bat dau...
Job ID a254a42c-4331-45db-b37c-70bb57f1cb57 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [micro-sent]  0.10 GB thuc · chay     4s · tai    2s ·   118,382 dong
  [micro-result(+72h)] uoc quet 0.08 GB, bat dau...
Job ID 9fb10b0d-d760-4fcc-90aa-edef5039d155 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [micro-result(+72h)]  0.08 GB thuc · chay     4s · tai    2s · 1,559,318 dong

organism vocab: top 30 khop tren 388,474 dong TRAIN
   phu 93.3% so lan mo tren train
study co ban ghi vi sinh: 147,001 / 227,835 (64.5%)


## C8 · Ghép + cột `*_mask` + ba tier

Mọi giá trị thiếu **giữ NaN và thêm cột chỉ báo**, không điền trung bình — nếu điền, "không được đo"
biến thành một giá trị lâm sàng giả, và mô hình học nhầm nó thành tín hiệu.

Ba tier trả lời ba câu tách bạch:

```
Q_CTX   boi canh + missingness + dem do luong + age/gender    (KHONG gia tri lam sang nao)
Q_CLIN  Q_CTX + gia tri labs (3 cua so) + vitals + comorbidity (KHONG mot cot vi sinh nao)
Q_FULL  Q_CLIN + vi sinh + one-hot organism
```

Khoảng cách `Q_CLIN − Q_CTX` là thứ đáng quan tâm: nếu ≈ 0 thì EHR chỉ đang đọc ra *loại bệnh nhân*,
không đọc ra *bệnh*.

In [12]:
DF = BASE[['study_id','subject_id','split','setting','index_time',
           'cur_hadm','cur_edstay']].copy()
DF['ctx_is_inpatient'] = (DF.setting == 'inpatient').astype('int8')
DF['ctx_is_ed']        = (DF.setting == 'ed').astype('int8')

# --- labs: pivot ba cua so thanh ba nhom cot ---
_VAL_TAGS = {t for t, _ in LAB_VALUE_WINDOWS}
for tag, _ in LAB_WINDOWS:
    for kind, pref in [('v', ''), ('dh', 'dh_'), ('n', 'n_')]:
        if kind == 'v' and tag not in _VAL_TAGS:
            continue                      # 90d: bo GIA TRI, giu dem + do moi
        p = labq.pivot(index='study_id', columns='canon', values=f'{kind}_{tag}')
        p.columns = [f'{pref}{c}_{tag}' for c in p.columns]
        DF = DF.merge(p.reset_index(), on='study_id', how='left')
DF = DF.merge(vitq, on='study_id', how='left').merge(como, on='study_id', how='left')
DF = DF.merge(demo, on='study_id', how='left').merge(micro, on='study_id', how='left')

# --- dem/chi bao: thieu nghia la KHONG, dien 0 la dung ---
for c in [c for c in DF.columns if c.startswith('como_')]:
    DF[c] = DF[c].fillna(0).astype('int16')
for c in ['micro_pos_n','micro_n_org','micro_sent_n','micro_sent_n_spec','vital_n_meas'] + \
         [c for c in DF.columns if c.startswith('micro_org_')] + \
         [c for c in DF.columns if c.startswith('n_lab_')]:
    if c in DF: DF[c] = DF[c].fillna(0).astype('int16')

# --- gia tri: GIU NaN + them cot mask ---
VALUE_COLS = ([c for c in DF.columns if re.match(r'lab_\w+_(24h|7d)$', c)]
              + [c for c in DF.columns if c.startswith('vital_') and c != 'vital_n_meas']
              + ['age'])
for c in VALUE_COLS:
    DF[f'{c}_mask'] = DF[c].notna().astype('int8')
DF['gender'] = DF.gender.map({'M': 0, 'F': 1})
DF['gender_mask'] = DF.gender.notna().astype('int8')

# --- ba tier, tach bang QUY TAC chu khong bang tay ---
MICRO_RE = re.compile(r'^micro_')
VAL_RE   = re.compile(r'^(lab_\w+_(24h|7d)$|vital_(temp_c|hr|rr|spo2|sbp|dbp)$|como_|age$|gender$)')
ID_COLS  = ['study_id','subject_id','split','setting','index_time','cur_hadm','cur_edstay']
FEAT     = [c for c in DF.columns if c not in ID_COLS]
Q_CTX    = [c for c in FEAT if not VAL_RE.match(c) and not MICRO_RE.match(c)]
Q_CLIN   = [c for c in FEAT if not MICRO_RE.match(c)]
Q_FULL   = FEAT
assert set(Q_CTX) < set(Q_CLIN) < set(Q_FULL)
_forbid = [c for c in Q_CTX if VAL_RE.match(c)]
assert not _forbid, f'Q_CTX ro ri gia tri lam sang: {_forbid}'
print(f'Q_CTX {len(Q_CTX)} | Q_CLIN {len(Q_CLIN)} | Q_FULL {len(Q_FULL)} cot')
print(f'  vi sinh chiem {len(Q_FULL)-len(Q_CLIN)} cot ({(len(Q_FULL)-len(Q_CLIN))/len(Q_FULL):.0%})')

Q_CTX 157 | Q_CLIN 212 | Q_FULL 247 cot
  vi sinh chiem 35 cot (14%)


/tmp/ipykernel_4848/523165362.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  DF[f'{c}_mask'] = DF[c].notna().astype('int8')
/tmp/ipykernel_4848/523165362.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  DF[f'{c}_mask'] = DF[c].notna().astype('int8')
/tmp/ipykernel_4848/523165362.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

## C9 · Kiểm chứng + ghi

In [13]:
assert len(DF) == len(BASE) and DF.study_id.is_unique
assert (DF.study_id.to_numpy() == BASE.study_id.to_numpy()).all(), 'thu tu study bi doi'

# --- BAT BIEN 2: khong mot gia tri nao den tu tuong lai ---
for c in [c for c in DF.columns if c.startswith('dh_lab_')] + ['vital_delta_h']:
    v = DF[c].dropna()
    assert (v >= 0).all(), f'LEAK: {c} co gia tri am -> do SAU index_time'
assert (DF.micro_sent_delta_h.dropna() >= 0).all(), 'LEAK: micro_sent sau index_time'
print('bat bien 2: labs/vitals/micro_sent deu <= index_time')
print(f'bat bien ngoai le: micro ket qua dung cua so [-{MICRO_PAST_D}d, +{MICRO_FUTURE_H}h]')

print('\n=== DO PHU THEO SETTING (doc bang nay truoc moi ket luan EHR) ===')
_L24 = [c for c in DF.columns if re.match(r'lab_\w+_24h$', c)]
_N90 = [c for c in DF.columns if c.startswith('n_lab_') and c.endswith('_90d')]
# 90d KHONG co cot gia tri (chi DEM + DO MOI) -> phai do bang cot dem.
# Do bang `lab_*_90d` se luon ra 0% vi khong cot nao khop.
assert _N90, 'khong thay cot n_lab_*_90d'
COV = DF[['setting']].copy()
# tach `no_ehr`: subject khong co trong hosp.patients -> khong co GI ca, ke ca age.
# Gop chung vao `outpatient` se giau mat mot quan the ~5% hoan toan khac.
COV['setting'] = np.where(DF.age.isna(), 'no_ehr', DF.setting)
COV['co_lab_24h']     = DF[_L24].notna().any(axis=1)
COV['co_lab_90d']     = DF[_N90].sum(axis=1) > 0
COV['co_vital']       = DF.vital_n_meas.gt(0)
COV['co_comorbidity'] = DF.como_n_prior_adm.gt(0)
COV['co_age']         = DF.age.notna()
COV['co_micro']       = DF.micro_sent_n.gt(0)
_ORD = ['inpatient', 'ed', 'outpatient', 'no_ehr']
rep = COV.groupby('setting').mean().reindex(_ORD).dropna(how='all')
rep.insert(0, 'n_study', COV.groupby('setting').size().reindex(rep.index).astype('Int64'))
print(rep.to_string(formatters={c: '{:.2%}'.format for c in rep.columns if c != 'n_study'}))

P = os.path.join(OUT_DIR, 'cxrlt_ehr.parquet')
DF.to_parquet(P, index=False)
json.dump(dict(Q_CTX=Q_CTX, Q_CLIN=Q_CLIN, Q_FULL=Q_FULL, labs=LABS,
               lab_windows=[[t, h] for t, h in LAB_WINDOWS],
               lab_value_windows=[[t, h] for t, h in LAB_VALUE_WINDOWS],
               micro_past_d=MICRO_PAST_D, micro_future_h=MICRO_FUTURE_H,
               org_vocab=[int(o) for o in ORG_VOCAB],
               comorbidity=list(COMO), vital_window_h=VITAL_WINDOW_H),
          open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json'), 'w'), indent=1)
print(f'\nghi {P}  ({os.path.getsize(P)/1e6:.1f} MB, {len(DF):,} study x {len(DF.columns)} cot)')
print(f'ghi cxrlt_ehr_spec.json  (dinh nghia 3 tier + vocab + cua so)')

print(f'''
================================================================
STAGE 1b XONG

TIEP THEO -> 30_cxrlt_ehr_only.ipynb
  XGB moi nhan mot model, fit train -> cham val, cho CA BA tier.
  BAT BUOC: AUROC + AP kem n_pos tung nhan, tach theo tier head/medium/tail
            VA tach theo setting. Kem shuffle control.
  Sweep MICRO_FUTURE_H {{0,24,48,72}}: tang deu = tin hieu nuoi cay that;
  nhay vot o 24h roi phang = dang bat hanh vi chi dinh.
================================================================''')